In [1]:
from llava.model.builder import load_pretrained_model
import argparse
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path

model_path = "llava-fastvithd_0.5b_stage3"
model_name = get_model_name_from_path(model_path)

parser = argparse.ArgumentParser()
parser.add_argument("--model-path", type=str, default="./llava-v1.5-13b")
parser.add_argument("--model-base", type=str, default=None)
parser.add_argument("--image-file", type=str, default=None, help="location of image file")
parser.add_argument("--prompt", type=str, default="Describe the image.", help="Prompt for VLM.")
parser.add_argument("--conv-mode", type=str, default="mistral")
parser.add_argument("--temperature", type=float, default=0.2)
parser.add_argument("--top_p", type=float, default=None)
parser.add_argument("--num_beams", type=int, default=1)
args, unknown = parser.parse_known_args()

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path, args.model_base, model_name, device="cuda"
)

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "C:\Users\kumar\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "C:\Users\kumar\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [6]:
import sys
print(sys.executable)

C:\Users\kumar\AppData\Local\Programs\Python\Python312\python.exe


In [7]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [9]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "C:\Users\kumar\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [8]:
import torch
print("Torch loaded:", torch.__version__)

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "C:\Users\kumar\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
device = "cuda"

In [ ]:
import os
import sys

# 1. Manually add the Torch library folder to the DLL search path
torch_dll_path = r"C:\Users\kumar\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\lib"
if os.path.exists(torch_dll_path):
    os.add_dll_directory(torch_dll_path)

# 2. Add the standard Python Scripts folder (for CUDA dependencies)
scripts_path = r"C:\Users\kumar\AppData\Local\Programs\Python\Python312\Scripts"
if os.path.exists(scripts_path):
    os.add_dll_directory(scripts_path)

# Now try the import
import torch
print(f"Success! Torch version: {torch.__version__}")
print(f"RTX 4050 detected: {torch.cuda.is_available()}")

In [ ]:
llm_model = model.model
print(llm_model)

In [ ]:
import torch
import torch.nn as nn

class MobileVLM(nn.Module):
    def __init__(self, vision_tower, mm_projector, llm_model):
        super().__init__()
        self.vision_tower = vision_tower
        self.mm_projector = mm_projector
        self.llm_model = llm_model

    def forward(self, pixel_values, input_ids, attention_mask=None):
        # 1. Vision Forward: FastViT features
        # FastViT output typically has a shape like [batch, channels, h, w] 
        # but your projector expects 3072 features.
        image_outputs = self.vision_tower(pixel_values) 
        
        # 2. Project Vision to LLM Space (3072 -> 896)
        # image_features shape: [batch, num_patches, 896]
        image_features = self.mm_projector(image_outputs)
        
        # 3. Get Text Embeddings (896)
        inputs_embeds = self.llm_model.embed_tokens(input_ids)
        
        # 4. Concatenate Vision + Text
        # The VLM reads the image tokens first, then the text
        combined_embeds = torch.cat([image_features, inputs_embeds], dim=1)
        
        # 5. LLM Forward with GQA (Native Qwen2Attention)
        # Note: Ensure attention_mask is adjusted for the added image tokens
        outputs = self.llm_model(
            inputs_embeds=combined_embeds,
            attention_mask=attention_mask
        )
        
        return outputs.logits

In [ ]:
llm_model.vision_tower = None
llm_model.mm_projector = None

In [ ]:
llm_model

In [ ]:
import torch
from transformers import BitsAndBytesConfig

# You would typically use this when loading from HuggingFace
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
torch.__version__

In [ ]:
import torch
import torchao
print(f"Torch: {torch.__version__}")
print(f"AO: {torchao.__version__}")